# Chapter 12 &mdash; Bound the Stack and the Model Learns It

**Concept 14 of the Chapter 12 decomposition:** *Bound the Stack and the Model Learns It*

Cap the nesting depth at two and the forbidden-symbol mass collapses &mdash; because a bounded stack was never a stack.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter12-PDA/Concept-Bounding-The-Stack/Concept-Bounding-The-Stack.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_PDA        import *
from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


Two notebooks of negative results earn one positive one.

The claim so far has been that the model fails on brackets **because the language
needs unbounded memory**. That is a claim about the language, not about the model, and
there is a clean way to test it: change the language so it no longer needs unbounded
memory, change nothing else, and see whether the failure goes away.

`DEPTH2` below is balanced brackets with the nesting **capped at two**. It is written
as a PDA, it uses the stack, `run_pda` runs it &mdash; but a stack that never holds
more than two symbols is a stack you could have replaced with three states. The
language is **regular**.

If the diagnosis is right, the same model on this language should behave like
Chapter 6: the forbidden-symbol mass should go towards zero rather than stalling
around a quarter.

It does. That is the notebook.

## 2. Definitions

### The model, the training loop, the sampler

In [ ]:
#@title minimal GPT implementation in PyTorch  (Andrej Karpathy)
#
# Read it, change it, break it.  This is the whole model: an embedding, a
# few attention blocks, a linear head.  Nothing here knows about automata.
""" super minimal decoder-only gpt """

import math
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F

class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        # regularization
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                    .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)

        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        q, k ,v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        # manual implementation of attention
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side

        # output projection
        y = self.c_proj(y)
        return y

class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.nonlin = nn.GELU()

    def forward(self, x):
        x = self.c_fc(x)
        x = self.nonlin(x)
        x = self.c_proj(x)
        return x

class Block(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

@dataclass
class GPTConfig:
    # these are default GPT-2 hyperparameters
    block_size: int = 1024
    vocab_size: int = 50304
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    bias: bool = False

class GPT(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight # https://paperswithcode.com/method/weight-tying

        # init all weights
        self.apply(self._init_weights)
        # apply special scaled init to the residual projections, per GPT-2 paper
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))

        # report number of parameters
        print("number of parameters: %d" % (sum(p.nelement() for p in self.parameters()),))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, f"Cannot forward sequence of length {t}, block size is only {self.config.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=device).unsqueeze(0) # shape (1, t)

        # forward the GPT model itself
        tok_emb = self.transformer.wte(idx) # token embeddings of shape (b, t, n_embd)
        pos_emb = self.transformer.wpe(pos) # position embeddings of shape (1, t, n_embd)
        x = tok_emb + pos_emb
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x[:, -1, :]) # note: only returning logits at the last time step (-1), output is 2D (b, vocab_size)
        return logits

&nbsp;

In [ ]:
def make_XY(seq, context_length):
    X, Y = [], []
    for i in range(len(seq) - context_length):
        X.append(seq[i:i + context_length])
        Y.append(seq[i + context_length])
    return (torch.tensor(X, dtype=torch.long),
            torch.tensor(Y, dtype=torch.long))

def train_gpt(gpt, X, Y, iters=200, lr=1e-3, every=20):
    optimizer = torch.optim.AdamW(gpt.parameters(), lr=lr, weight_decay=1e-1)
    losses = []
    for i in range(iters):
        logits = gpt(X)
        loss = F.cross_entropy(logits, Y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        losses.append(loss.item())
        if i % every == 0 or i == iters - 1:
            print(i, loss.item())
    return losses

&nbsp;

In [ ]:
# --- sample from the model, exactly as Karpathy does --------------------
def sample(gpt, start, steps=24):
    xi = list(start)
    full = xi.copy()
    for _ in range(steps):
        x = torch.tensor(xi, dtype=torch.long)[None, ...]
        probs = nn.functional.softmax(gpt(x), dim=-1)
        t = torch.multinomial(probs[0], num_samples=1).item()
        xi = xi[1:] + [t]
        full.append(t)
    return ''.join(map(str, full))

### The corpus, from a PDA

In [ ]:
# --- the training data: strings the PDA accepts, run together ------------
from functools import reduce

def pda_accepts(P, s, STKMAX=20):
    surv, paths, visited = run_pda(s, P, acceptance='ACCEPT_F',
                                   STKMAX=STKMAX, chatty=False)
    return len(paths) > 0        # accepted iff SOME path ended in F

def corpus_from(P, upto=16000, STKMAX=8):
    strings = [nthnumeric(i, ['0', '1']) for i in range(upto)]
    good = [s for s in strings if pda_accepts(P, s, STKMAX)]
    return good, list(map(int, reduce(lambda a, b: a + b, good)))

### Two languages: one needs a stack, one only looks like it

In [ ]:
# --- the language: balanced brackets, with 0 for '(' and 1 for ')' -------
# The stack is doing the one thing no DFA can do: counting with no bound.
DYCK = md2mc('''PDA
IF : 0 , # ; 0# -> A
A  : 0 , 0 ; 00 -> A
A  : 1 , 0 ; '' -> A
A  : '' , # ; # -> IF
''')

# --- the same language, but the nesting may never go deeper than 2 -------
# Still written as a PDA -- but a stack that can only ever hold two things
# is a stack you could replace with three states.  This language is
# REGULAR, and that is the whole point of the notebook.
DEPTH2 = md2mc('''PDA
IF : 0 , # ; 0# -> D1
D1 : 0 , 0 ; 00 -> D2
D1 : 1 , 0 ; '' -> D0
D2 : 1 , 0 ; '' -> D1
D0 : 0 , # ; 0# -> D1
D0 : '' , # ; # -> IF
''')

# --- how much probability does the model give a FORBIDDEN symbol? --------
# Where the stack is empty the language cannot close, so '1' is forbidden.
# Where a capped stack is full it cannot open, so '0' is forbidden.  Ask
# the model how much mass it puts there.  No threshold and no score: one
# number, between 0 and 1, saying how much of the rule it has picked up.
def forbidden_mass(gpt, k, good, cap=None):
    total, n = 0.0, 0
    for s in good:
        depth = 0
        for i, c in enumerate(s):
            bad = 1 if depth == 0 else (0 if depth == cap else None)
            if bad is not None and i >= k:
                x = torch.tensor([int(ch) for ch in s[i - k:i]],
                                 dtype=torch.long)[None, ...]
                p = nn.functional.softmax(gpt(x), dim=-1)[0].tolist()
                total += p[bad]
                n += 1
            depth += 1 if c == '0' else -1
    return total / max(n, 1), n

<!-- nav-strip -->

---

&larr;&nbsp;[Ch12&nbsp;13.&nbsp;More Context Is Not a Stack](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter12-PDA/Concept-More-Context-Is-Not-A-Stack/Concept-More-Context-Is-Not-A-Stack.ipynb) &nbsp;&middot;&nbsp; [**Chapter 12** index](https://github.com/ganeshutah/Jove/blob/master/Chapter12-PDA/README.md) &nbsp;&middot;&nbsp; [Ch13&nbsp;1.&nbsp;Turing's Definition of Computation, and the Entscheidungsproblem](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter13-TM/Concept-Turings-Definition/Concept-Turings-Definition.ipynb)&nbsp;&rarr;

---

## 3. Tests

First check that `DEPTH2` is the language it claims to be.

In [ ]:
def depth_ok(s, cap=2):
    depth = 0
    for c in s:
        depth += 1 if c == '0' else -1
        if depth < 0 or depth > cap: return False
    return depth == 0

from itertools import product
cases = [''.join(p) for n in range(11) for p in product('01', repeat=n)]
bad = [s for s in cases if pda_accepts(DEPTH2, s, STKMAX=6) != depth_ok(s)]
print('disagreements over %d strings :' % len(cases), bad)
assert not bad
print('DEPTH2 is exactly "balanced, and never nested more than 2 deep"')

It is regular, and here is the witness: three states are enough to track the depth, so no stack is needed at all.

In [ ]:
D2 = md2mc('''DFA
IF : 0 -> A1
IF : 1 -> DEAD
A1 : 0 -> A2
A1 : 1 -> IF
A2 : 0 -> DEAD
A2 : 1 -> A1
DEAD : 0|1 -> DEAD
''')
bad = [s for s in cases if accepts_dfa(D2, s) != depth_ok(s)]
print('the DFA disagrees with the PDA on :', bad)
assert not bad
dotObj_dfa(min_dfa(D2), FuseEdges=True)

Two corpora, same recipe.

In [ ]:
goodD, seqD = corpus_from(DYCK)
good2, seq2 = corpus_from(DEPTH2, upto=70000, STKMAX=6)
print('unbounded nesting : %3d strings, %d symbols' % (len(goodD), len(seqD)))
print('nesting capped at 2: %3d strings, %d symbols' % (len(good2), len(seq2)))

**The comparison.** Same model, same training, same measurement &mdash; the language is the only thing that differs.

In [ ]:
def run(seq, good, k, cap=None, iters=200):
    X, Y = make_XY(seq, k)
    config = GPTConfig(block_size=k, vocab_size=2, n_layer=4, n_head=4,
                       n_embd=16, bias=False)
    torch.manual_seed(1337)
    gpt = GPT(config)
    losses = train_gpt(gpt, X, Y, iters=iters, every=iters)
    mass, spots = forbidden_mass(gpt, k, good, cap)
    return gpt, losses[-1], mass

print('%-22s %-6s %-9s %s' % ('language', 'window', 'loss', 'forbidden mass'))
for k in (3, 5, 8):
    _, fl, m = run(seqD, goodD, k, None)
    print('%-22s k=%-4d %-9.4f %.3f' % ('unbounded nesting', k, fl, m))
for k in (3, 5, 8):
    _, fl, m = run(seq2, good2, k, 2)
    print('%-22s k=%-4d %-9.4f %.3f' % ('nesting capped at 2', k, fl, m))

The two blocks are the result.

In [ ]:
print("Unbounded nesting: the mass comes down from a half and stops, around")
print("a quarter, and widening the window only inches it along.")
print()
print("Capped nesting: the same mass drops by a factor of several, and keeps")
print("dropping as the window widens.  The model is closing the leak.")
print()
print("Nothing about the model changed between the two blocks.  The stack")
print("was there in both PDAs.  What changed is whether the stack was DOING")
print("anything a window could not do.")

Which is Concept 5 of this chapter, and the pumping lemma, from a third direction.

In [ ]:
print("A PDA whose stack is bounded by a constant recognises a REGULAR")
print("language: put the stack contents into the state, and the states are")
print("still finitely many.  That is a theorem, and DEPTH2 is an instance.")
print()
print("So the honest statement of all three notebooks is not")
print("    'transformers cannot do context-free languages'")
print("but")
print("    'a fixed context window is a finite memory, and a finite memory")
print("     recognises exactly the regular languages.'")
print()
print("The first is a claim about an architecture and would need arguing.")
print("The second is Chapter 6, and you already proved it.")

## 4. Animation

The bounded machine, whose stack you could throw away.

In [ ]:
from jove.AnimatePDA import *
AnimatePDA(DEPTH2, FuseEdges=True)

## 5. Exercises


1. Change the cap to 3 by adding a `D3` state to the PDA. Does the forbidden mass at
   `k=3` get better or worse, and why would you expect that direction?
2. Put the stack contents into the state by hand for `DEPTH2` and draw the DFA. How
   many states did you need, and how does that scale with the cap?
3. The capped corpus needed `upto=70000` to find as many strings. Why is the capped
   language so much sparser, and does that sparsity help or hurt the model?
4. Sample from the capped model with `try_samples(gpt, k, DEPTH2, length=32)`. Compare
   the acceptance rate with the same test on `DYCK` in the previous notebook.
5. For which of these two languages would Chapter 6's `plot_model` picture be an
   honest drawing of the language? Justify with the number of states.

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter12-PDA/Concept-Bounding-The-Stack')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')